# HydroSight TN - XGBoost Training

This notebook builds the borewell training table, trains the classifier, exports SHAP summaries, and predicts the full groundwater potential map.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src import FEATURE_ORDER
from src.train import build_training_dataset, generate_shap_plot, predict_full_map, train_xgboost

classified_dir = project_root / 'data' / 'classified'
raw_dir = project_root / 'data' / 'raw'
output_dir = project_root / 'outputs'
model_dir = project_root / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

classified_layers = {feature: classified_dir / f'{feature}.tif' for feature in FEATURE_ORDER}
training_csv = build_training_dataset(
    raw_dir / 'borewells.shp',
    classified_layers,
    ahp_score_raster=output_dir / 'gwpz_ahp_score.tif',
    output_csv=output_dir / 'training_data.csv',
)
training_csv

In [ ]:
model, training_df = train_xgboost(training_csv, model_path=model_dir / 'gwpz_xgboost.pkl')
generate_shap_plot(model, training_df[FEATURE_ORDER], FEATURE_ORDER, out_path=output_dir / 'shap_summary.png')
predict_full_map(model, classified_layers, output_dir / 'gwpz_xgboost.tif', probability_path=output_dir / 'gwpz_xgboost_prob.tif')